Cell 1 -Authentication to Source Blob Storage


In [0]:
SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

# Cell 2 — Read load_type Job parameter and set paths


In [0]:
from datetime import datetime, timezone, timedelta

# Read load_type from Databricks Job widget parameter.
# Default: "incremental" so the notebook is safe to run manually without setting the widget.
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
load_type = dbutils.widgets.get("load_type").strip().lower()

if load_type not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type='{load_type}'. Must be 'full' or 'incremental'.")

print(f"load_type : {load_type}")

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev_7405618615455893/bronze/bronze_volume"
BASE_SUBPATH  = "invoices"

now = datetime.now(timezone.utc)

if load_type == "full":
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/"
    print(f"Mode : full — all invoice dates")
else:
    # Incremental: target yesterday's completed date (job fires at 01:00 UTC daily)
    yesterday   = now - timedelta(days=1)
    partition   = yesterday.strftime("%Y/%m/%d")
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/"
    print(f"Mode : incremental — {partition}  (yesterday UTC)")

print(f"Source : {SOURCE_PATH}")
print(f"Bronze : {BRONZE_PATH}")

# Cell 3 — List source PDF files

In [0]:
def list_files_recursive(path):
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return []
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

source_files = list_files_recursive(SOURCE_PATH)
pdf_files    = [f for f in source_files if f.name.endswith(".pdf")]

total_size_mb = sum(f.size for f in pdf_files) / (1024 * 1024)
print(f"PDF files found : {len(pdf_files)}")
print(f"Total size      : {round(total_size_mb, 1)} MB")
print()
for f in pdf_files[:20]:   # preview first 20
    rel = f.path.replace(SOURCE_PATH, "")
    print(f"  {rel:<60}  [{round(f.size/1024, 1)} KB]")
if len(pdf_files) > 20:
    print(f"  ... and {len(pdf_files) - 20} more")

# Cell 4 — Copy PDF files to Bronze Volume


In [0]:
copied  = []
skipped = []

for file_info in pdf_files:
    relative_path = file_info.path.replace(SOURCE_PATH, "")
    dest_path     = BRONZE_PATH + relative_path
    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        if len(copied) <= 20 or len(copied) % 50 == 0:
            print(f"  COPIED  {relative_path}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {relative_path} — {e}")

print(f"\nResult: {len(copied)} copied, {len(skipped)} failed")
if skipped:
    raise Exception(f"{len(skipped)} file(s) failed — check output above.")

# Cell 5 — Verify files in Bronze Volume

In [0]:
bronze_files = list_files_recursive(BRONZE_PATH)
bronze_pdfs  = [f for f in bronze_files if f.name.endswith(".pdf")]

status = "PASS" if len(bronze_pdfs) == len(pdf_files) else "FAIL"
print(f"[{status}] Source: {len(pdf_files)} PDFs  →  Bronze: {len(bronze_pdfs)} PDFs")

assert len(bronze_pdfs) == len(pdf_files), (
    f"Count mismatch — source: {len(pdf_files)}, bronze: {len(bronze_pdfs)}"
)
print("Verification passed — all PDFs confirmed in Bronze Volume.")
     

# Cell 6 — Invoice metadata from file names

In [0]:
import re
from pyspark.sql import Row

rows = []
for f in bronze_pdfs:
    # Extract partition from path: .../invoices/YYYY/MM/DD/INV-AU-YYYY-NNNN.pdf
    m = re.search(r"invoices/(\d{4})/(\d{2})/(\d{2})/(.+\.pdf)$", f.path)
    if m:
        rows.append(Row(
            invoice_id  = m.group(4).replace(".pdf", ""),
            year        = m.group(1),
            month       = m.group(2),
            day         = m.group(3),
            file_size_kb= round(f.size / 1024, 1),
            bronze_path = f.path
        ))

df_meta = spark.createDataFrame(rows)
print(f"Invoice metadata rows: {df_meta.count():,}")
display(df_meta.orderBy("year", "month", "day").limit(20))

In [0]:
print("=" * 60)
print("BRONZE INVOICE PDF MIGRATION — RUN SUMMARY")
print("=" * 60)
print(f"load_type       : {load_type}")
if load_type == "incremental":
    print(f"Date (UTC-1d)   : {partition}")
print(f"Source path     : {SOURCE_PATH}")
print(f"Bronze path     : {BRONZE_PATH}")
print(f"PDFs copied     : {len(copied)}")
print(f"PDFs failed     : {len(skipped)}")
print(f"Bronze total    : {len(bronze_pdfs)}")
print("=" * 60)
print("Next step: Silver layer parses PDF metadata or content for invoice Delta table.")